# gbm_os — Usage Guide

End-to-end walkthrough of the `gbm_os` cohort-access package: loading the manifest,
exploring the data, selecting a study cohort, and feeding it to PyTorch / MONAI / torchio.

**Prerequisites**
- `output/master_manifest.csv` built by `gbm-manifest build`
- Raw datasets mounted at `/mnt/disk1/datasets/`
- `source activate.sh` (or venv active) so `gbm_os` and `gbm_manifest` are importable
- **Framework backends** require PyTorch — install torch with your CUDA build first:
  `pip install torch --index-url https://download.pytorch.org/whl/cu121`
  then: `pip install -e ".[all]"`  (or individually: `[monai]`, `[torchio]`, `[torch]`)

The core install (`pip install -e .`) is pandas + numpy + pydantic only — enough to
select cohorts and resolve paths, with no loading backend.

## 1. Load the Cohort

In [1]:
from pathlib import Path

from gbm_os import Cohort
from gbm_os.studies import GBM_OS_STUDY

DATA_ROOTS = {
    "brats2020": "/mnt/disk1/datasets/BraTS-2020",
    "rhuh_gbm":  "/mnt/disk1/datasets/RHUH-GBM",
    "upenn_gbm": "/mnt/disk1/datasets/UPENN-GBM",
    "ucsf_pdgm": "/mnt/disk1/datasets/UCSF-PDGM",
}
MANIFEST = Path("../output/master_manifest.csv")

# The study supplies the cohort policies — survival thresholds, held-out cohort,
# duplicate priority — so they cannot drift from the definition you publish.
config = GBM_OS_STUDY.config(DATA_ROOTS)

cohort = Cohort.from_manifest(MANIFEST, data_roots=DATA_ROOTS, config=config)
print(f"Manifest loaded: {len(cohort.select())} sessions")

Manifest loaded: 1657 sessions


## 2. Explore with cohort_summary

Before selecting anything, get a full picture of what is in the manifest: demographics, survival, modality completeness, and clinical variable distributions.

In [2]:
from gbm_os import cohort_summary

full_view = cohort.select()
s = cohort_summary(full_view)
print(s.report())

════════════════════════════════════════════════════════════════════════
  COHORT SUMMARY   1536 patients · 1657 sessions · 4 datasets
════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────
  Demographics & survival  (patient-level)
────────────────────────────────────────────────────────────────────────
           n_pat  n_ses  n_long  age_μ  age_med  age_σ  age_miss%   os_μ  os_med   os_σ  os_miss%  event%
dataset                                                                                                  
brats2020    369    369       0   61.2     61.5   11.9       36.0  445.5   369.0  355.2      36.0    99.6
rhuh_gbm      40    120      40   63.0     64.0    9.2        0.0  437.4   364.0  272.4       0.0    77.5
ucsf_pdgm    501    501       0   56.9     59.0   15.0        0.0  575.3   421.0  517.1       0.2    50.1
upenn_gbm    626    667      41   62.6     63.4   12.5        0.0  511.6

In [3]:
# Access individual tables as DataFrames
s.clinical       # patient-level demographics & survival

,n_patients,n_sessions,n_longitudinal,age_mean,age_median,age_std,age_pct_missing,os_days_mean,os_days_median,os_days_std,os_days_pct_missing,event_rate_pct
dataset,,,,,,,,,,,,
brats2020,369,369,0,61.2,61.5,11.9,36.0,445.5,369.0,355.2,36.0,99.6
rhuh_gbm,40,120,40,63.0,64.0,9.2,0.0,437.4,364.0,272.4,0.0,77.5
ucsf_pdgm,501,501,0,56.9,59.0,15.0,0.0,575.3,421.0,517.1,0.2,50.1
upenn_gbm,626,667,41,62.6,63.4,12.5,0.0,511.6,384.0,519.3,4.3,97.3
TOTAL,1536,1657,81,60.3,61.4,13.5,8.7,521.3,393.0,490.3,10.5,80.2


In [4]:
s.imaging        # session-level modality completeness

,n_sessions,pct_t1,pct_t1ce,pct_t2,pct_flair,pct_seg,pct_complete
dataset,,,,,,,
brats2020,369,100.0,100.0,100.0,100.0,99.7,100.0
rhuh_gbm,120,100.0,100.0,100.0,100.0,100.0,100.0
ucsf_pdgm,501,100.0,100.0,100.0,100.0,100.0,100.0
upenn_gbm,667,79.0,77.5,77.8,77.4,73.9,36.0
TOTAL,1657,91.6,90.9,91.1,90.9,89.4,74.2


In [5]:
s.distributions["eor"]    # EOR distribution per dataset

,brats2020,rhuh_gbm,ucsf_pdgm,upenn_gbm,TOTAL
eor,,,,,
missing,133 (36.0%),0 (0.0%),0 (0.0%),0 (0.0%),133 (8.7%)
GTR,119 (32.2%),27 (67.5%),248 (49.5%),359 (57.3%),753 (49.0%)
unknown,107 (29.0%),13 (32.5%),1 (0.2%),57 (9.1%),178 (11.6%)
STR,10 (2.7%),0 (0.0%),198 (39.5%),0 (0.0%),208 (13.5%)
biopsy,0 (0.0%),0 (0.0%),54 (10.8%),0 (0.0%),54 (3.5%)
non_GTR,0 (0.0%),0 (0.0%),0 (0.0%),210 (33.5%),210 (13.7%)


In [6]:
# Optional: read NIfTI headers to check volume shape and voxel spacing.
# Slower — one header read per session x modality. Use on a filtered view.
baseline_rhuh = cohort.select(baseline_only=True, datasets=["rhuh_gbm"])
s_spatial = cohort_summary(baseline_rhuh, scan_headers=True)
s_spatial.spatial

H_mean  H_median  H_std  W_mean  W_median  W_std  D_mean  \
dataset  modality                                                             
rhuh_gbm flair     239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         seg       239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         t1        239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         t1ce      239.75     240.0   1.58  239.75     240.0   1.58  154.57   
         t2        239.75     240.0   1.58  239.75     240.0   1.58  154.57   

                   D_median  D_std  spacing_x_mm_mean  spacing_x_mm_median  \
dataset  modality                                                            
rhuh_gbm flair        155.0   2.69                1.0                  1.0   
         seg          155.0   2.69                1.0                  1.0   
         t1           155.0   2.69                1.0                  1.0   
         t1ce         155.0   2.69                1.0                  1.0   
         t2           155.0   2.69                1.0                  1.0   

                   spacing_x_mm_std  spacing_y_mm_mean  spacing_y_mm_median  \
dataset  modality                                                             
rhuh_gbm flair                  0.0                1.0                  1.0   
         seg                    0.0                1.0                  1.0   
         t1                     0.0                1.0                  1.0   
         t1ce                   0.0                1.0                  1.0   
         t2                     0.0                1.0                  1.0   

                   spacing_y_mm_std  spacing_z_mm_mean  spacing_z_mm_median  \
dataset  modality                                                             
rhuh_gbm flair                  0.0                1.0                  1.0   
         seg                    0.0                1.0                  1.0   
         t1                     0.0                1.0                  1.0   
         t1ce                   0.0                1.0                  1.0   
         t2                     0.0                1.0                  1.0   

                   spacing_z_mm_std  n_scanned  
dataset  modality                               
rhuh_gbm flair                  0.0         40  
         seg                    0.0         40  
         t1                     0.0         40  
         t1ce                   0.0         40  
         t2                     0.0         40

## 3. Select a Study Cohort

A **study** is a declarative, versioned set of eligibility criteria. It lives in
`gbm_os/studies.py` rather than in your notebook or the pipeline, so the cohort you
report is the cohort the code selects.

Run `gbm-manifest studies` on the command line to list them.

In [7]:
print(GBM_OS_STUDY.describe())

gbm-os v2
Overall-survival classification over baseline preoperative GBM MRI, held out on UPENN-GBM.

Eligibility:
  - baseline session only (session_index == 0)
  - all four structural modalities present
  - eor == 'GTR'
  - who_grade in [4, None]
  - has_os == True

Policy:
  - os_class thresholds: <300 / <450 / >=
  - external cohort: ['upenn_gbm']
  - duplicate policy: drop (priority ['brats2020', 'ucsf_pdgm', 'rhuh_gbm', 'upenn_gbm'])


In [8]:
view = GBM_OS_STUDY.apply(cohort)

print(f"Selected: {len(view)} sessions")
print(cohort_summary(view).report())

Selected: 502 sessions
════════════════════════════════════════════════════════════════════════
  COHORT SUMMARY   502 patients · 502 sessions · 4 datasets
════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────
  Demographics & survival  (patient-level)
────────────────────────────────────────────────────────────────────────
           n_pat  n_ses  n_long  age_μ  age_med  age_σ  age_miss%   os_μ  os_med   os_σ  os_miss%  event%
dataset                                                                                                  
brats2020    119    119       0   62.0     63.4   12.0        0.0  445.7   374.0  343.9       0.0    99.2
rhuh_gbm      27     27       0   63.1     65.0    8.4        0.0  493.9   395.0  292.0       0.0    74.1
ucsf_pdgm    231    231       0   60.6     61.0   12.8        0.0  556.0   430.0  429.5       0.0    55.0
upenn_gbm    125    125       0   62.2     61.9   1

### Or build the criteria yourself

The study is just a convenience — every criterion is available directly on `select()`,
and all of them are optional and composable. Use this when exploring; use the study
when reporting.

Note `who_grade in [4, None]`: grade IV wherever grade is recorded. The `None` admits
UPENN, which ships no grade column because the cohort is GBM by construction, and
excludes BraTS LGG and UCSF grade II/III, which are recorded explicitly.

In [9]:
manual = cohort.select(
    baseline_only=True,          # one scan per patient (preop)
    require_complete=True,       # all four structural modalities present
    filters={
        "eor":       "GTR",      # gross total resection only
        "who_grade": [4, None],  # grade IV where grade is recorded
        "has_os":    True,       # survival data available
    },
    resolve_duplicates="drop",   # drop UPENN when a BraTS duplicate exists
)

assert len(manual) == len(view)   # same cohort, spelled out
print(f"{len(manual)} sessions")

502 sessions


### Where did everyone go?

Every view carries the criteria that produced it and every row they dropped. Criteria
are applied in order and each excluded session is attributed to the **first** one that
removed it, so the reasons partition the drops exactly — which is what makes the cohort
table defensible to a reviewer.

In [10]:
print(view.provenance())

input                      1661
baseline_only              1661 →   1521  (−140)
require_complete           1521 →   1126  (−395)
filter:eor                 1126 →    526  (−600)
filter:who_grade            526 →    509  (−17)
filter:has_os               509 →    503  (−6)
resolve_duplicates          503 →    502  (−1)
selected                    502


In [11]:
exclusions = view.exclusions()

print(f"{len(view)} selected + {len(exclusions)} excluded = "
      f"{len(view) + len(exclusions)}\n")
print(exclusions["exclusion_reason"].value_counts().to_string())

# Every dropped session, with its reason — this is your CONSORT diagram
exclusions[["dataset", "patient_id", "exclusion_reason"]].head()

502 selected + 1159 excluded = 1661

exclusion_reason
filter:eor            600
require_complete      395
baseline_only         140
filter:who_grade       17
filter:has_os           6
resolve_duplicates      1


,dataset,patient_id,exclusion_reason
0,rhuh_gbm,RHUH-0001,baseline_only
1,rhuh_gbm,RHUH-0001,baseline_only
2,rhuh_gbm,RHUH-0002,baseline_only
3,rhuh_gbm,RHUH-0002,baseline_only
4,rhuh_gbm,RHUH-0003,baseline_only


### Censoring is your decision, not the study's

The manifest records censored outcomes and the study keeps them. Whether your model may
use a patient who was still alive at last follow-up is a **modelling** choice, so it is
made here rather than baked into the cohort — filtering it earlier would make the
alternative unreachable.

`os_event`: `1` = death observed, `0` = censored (alive at last contact).

In [12]:
deceased = view.select(filters={"os_event": 1})

print(f"study cohort   : {len(view)}  (censoring intact)")
print(f"deceased only  : {len(deceased)}  (complete-case)")
print(f"censored cases : {len(view) - len(deceased)}\n")

# The narrowing is traced like any other criterion
print(deceased.provenance())

study cohort   : 502  (censoring intact)
deceased only  : 390  (complete-case)
censored cases : 112

input                       502
filter:os_event             502 →    390  (−112)
resolve_duplicates          390 →    390  (−0)
selected                    390


In [13]:
# Views can be further narrowed by chaining .select()
train_view    = view.select(partition="train")     # BraTS + RHUH + UCSF
external_view = view.select(partition="external")  # UPENN

print(f"Train: {len(train_view)}  |  External test: {len(external_view)}")

Train: 377  |  External test: 125


In [14]:
# .to_frame() gives the underlying DataFrame for inspection or export
df = view.to_frame()
df[["dataset", "patient_id", "age", "os_days", "os_class", "eor"]].head(10)

,dataset,patient_id,age,os_days,os_class,eor
0,brats2020,BraTS20_Training_001,60.463,289.0,0,GTR
1,brats2020,BraTS20_Training_002,52.263,616.0,2,GTR
2,brats2020,BraTS20_Training_003,54.301,464.0,2,GTR
3,brats2020,BraTS20_Training_004,39.068,788.0,2,GTR
4,brats2020,BraTS20_Training_005,68.493,465.0,2,GTR
5,brats2020,BraTS20_Training_006,67.126,269.0,0,GTR
6,brats2020,BraTS20_Training_007,69.912,503.0,2,GTR
7,brats2020,BraTS20_Training_009,56.419,1155.0,2,GTR
8,brats2020,BraTS20_Training_010,48.367,515.0,2,GTR
9,brats2020,BraTS20_Training_012,65.899,495.0,2,GTR


## 4. Iterate Directly (Framework-Agnostic)

Each iteration yields a `SampleSpec` — resolved absolute paths, all clinical facts, and per-sample imaging metadata.

In [15]:
spec = next(iter(view))

print("Key:              ", spec.global_session_key)
print("Dataset:          ", spec.dataset)
print("OS days / class:  ", spec.os_days, "/", spec.os_class)
print("EOR:              ", spec.eor)
print("IDH:              ", spec.idh_status)
print("Seg convention:   ", spec.seg_convention)          # 'brats_legacy' or 'rhuh'
print("Pre-normalised:   ", spec.intensity_prenormalised) # True only for RHUH
print()
print("Paths:")
for mod, path in spec.paths.items():
    print(f"  {mod:6s}  present={spec.present[mod]}  {path}")

Key:               brats2020__BraTS20_Training_001__tp0
Dataset:           brats2020
OS days / class:   289.0 / 0
EOR:               GTR
IDH:               None
Seg convention:    brats_legacy
Pre-normalised:    False

Paths:
  t1      present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t1.nii
  t1ce    present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t1ce.nii
  t2      present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_t2.nii
  flair   present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Training_001_flair.nii
  seg     present=True  /mnt/disk1/datasets/BraTS-2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001/BraTS20_Trai

## 5. PyTorch / Nibabel Backend

Loads volumes eagerly via nibabel. Returns raw float32 numpy arrays — **no intensity normalisation applied**. Apply `foreground_zscore` yourself, respecting the `intensity_prenormalised` flag per sample.

In [16]:
from gbm_os.transforms import foreground_zscore

dataset = view.to_torch(include_seg=True)
item = dataset[0]

print("image shape:        ", item["image"].shape)          # [C, H, W, D]
print("modality_mask:      ", item["modality_mask"])        # [C]  1=present, 0=zero-filled
print("seg shape:          ", item["seg"].shape)            # [1, H, W, D], RHUH already remapped
print("seg unique labels:  ", set(item["seg"].flatten().tolist()))
print("os_days / os_class: ", item["os_days"], "/", item["os_class"])

image shape:        

 (4, 240, 240, 155)
modality_mask:       [1. 1. 1. 1.]
seg shape:           (1, 240, 240, 155)


seg unique labels:   {0, 1, 2, 4}
os_days / os_class:  289.0 / 0


In [17]:
# Intensity normalisation — apply per channel, skip RHUH (already z-scored at source)
import numpy as np

spec  = next(iter(view))
image = item["image"]   # [C, H, W, D]

normalised = np.stack([
    foreground_zscore(image[c], intensity_prenormalised=spec.intensity_prenormalised)
    for c in range(image.shape[0])
])

print("Normalised image shape:", normalised.shape)
print("Ch-0 foreground mean (should be ~0):",
      normalised[0][normalised[0] > 0].mean().round(4))

Normalised image shape: (4, 240, 240, 155)
Ch-0 foreground mean (should be ~0): 0.7726


## 6. MONAI Backend

Hands file paths to a `monai.data.Dataset`. You supply the transform pipeline. Add `SegRemapd` **after** `LoadImaged` to remap RHUH seg labels — `seg_convention` is already in the data dict.

Install: `pip install gbm-manifest[monai]`

In [18]:
try:
    from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd
    from gbm_os.transforms import SegRemapd

    modalities = ["t1", "t1ce", "t2", "flair"]

    transforms = Compose([
        LoadImaged(keys=modalities + ["seg"]),
        EnsureChannelFirstd(keys=modalities + ["seg"]),
        SegRemapd(seg_key="seg"),   # reads seg_convention from dict, remaps RHUH 3->4
        # add your own spatial / intensity transforms here
    ])

    monai_ds = view.to_monai(transforms=transforms, include_seg=True)
    item = monai_ds[0]

    print("t1ce shape:              ", item["t1ce"].shape)
    print("seg shape:               ", item["seg"].shape)
    print("seg_convention:          ", item["seg_convention"])
    print("intensity_prenormalised: ", item["intensity_prenormalised"])

except ImportError:
    print("MONAI not installed. Run: pip install gbm-manifest[monai]")

MONAI not installed. Run: pip install gbm-manifest[monai]


## 7. torchio Backend

Wraps each session as a `tio.Subject`. Seg is **eagerly loaded and remapped** at construction — no extra transform needed.

Install: `pip install gbm-manifest[torchio]`

In [19]:
try:
    import torchio as tio

    tio_ds = view.to_torchio(include_seg=True)
    subject = tio_ds[0]

    print("t1ce shape:", subject["t1ce"].shape)   # [1, H, W, D]
    print("seg shape: ", subject["seg"].shape)
    print("os_days:   ", subject["os_days"])

    # Add spatial transforms via tio.Compose as usual
    spatial_transforms = tio.Compose([
        tio.RescaleIntensity(out_min_max=(0, 1)),
        tio.CropOrPad((240, 240, 155)),
    ])
    print("torchio dataset ready:", len(tio_ds), "subjects")

except ImportError:
    print("torchio not installed. Run: pip install gbm-manifest[torchio]")

torchio not installed. Run: pip install gbm-manifest[torchio]


## 8. Cross-Validation Splits

Stratified group k-fold: stratified by `os_class`, grouped by `patient_id` so no patient leaks across folds.

In [20]:
folds = train_view.split(k=5, seed=42)

for i in range(folds.k):
    tr = folds.fold(i, split="train")
    va = folds.fold(i, split="val")
    print(f"Fold {i}:  train={len(tr)}  val={len(va)}")

Fold 0:  train=296  val=81
Fold 1:  train=299  val=78
Fold 2:  train=304  val=73


Fold 3:  train=304  val=73
Fold 4:  train=305  val=72


In [21]:
# Each fold returns a CohortView — plug into any backend
fold0_train_ds = folds.fold(0, split="train").to_torch()
fold0_val_ds   = folds.fold(0, split="val").to_torch()

print("Fold 0 train:", len(fold0_train_ds))
print("Fold 0 val:  ", len(fold0_val_ds))

Fold 0 train: 296
Fold 0 val:   81


## 9. Age Normalisation (Leakage-Safe)

`AgeNormalizer` fits on the training fold only and applies those stats to validation — prevents leakage.

In [22]:
from gbm_os.transforms import AgeNormalizer

fold_train = folds.fold(0, split="train")
fold_val   = folds.fold(0, split="val")

norm = AgeNormalizer()
norm.fit(fold_train)

train_ages = norm.transform(fold_train)
val_ages   = norm.transform(fold_val)    # uses train stats — no leakage

print("Train age mean (normalised, should be ~0):", train_ages.mean().round(4))
print("Train age std  (normalised, should be ~1):", train_ages.std().round(4))
print("Val   age mean (will differ from 0):      ", val_ages.mean().round(4))

Train age mean (normalised, should be ~0): 0.0
Train age std  (normalised, should be ~1): 1.0
Val   age mean (will differ from 0):       -0.0903
